# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR² Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
**Croissant schema URL:**

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` and plotting libraries are installed
!pip install -U mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The main entities described in the Croissant schema are: **record sets** (tables), **fields** (logical columns), and **columns** (physical columns in files). All are referenced by their unique `@id`s.

In [ ]:
# List the available record sets using their @id
record_sets = list(dataset.record_sets)
print('Available record sets and their @id:')
for rs in record_sets:
    print(f"- {getattr(rs, '@id', None)}: {getattr(rs, 'name', None)}")

In [ ]:
# For each record set, list the fields (with @id and name)
for rs in record_sets:
    print(f"\nRecord Set: {getattr(rs, 'name', None)} (@id: {getattr(rs, '@id', None)})")
    try:
        fields = getattr(rs, 'fields', [])
        for field in fields:
            print(f"  - Field @id: {getattr(field, '@id', None)}, name: {getattr(field, 'name', None)}")
    except Exception as e:
        print('  No fields available or cannot extract fields.')

## 3. Data Extraction
We now load the tabular data from a specific record set (by its `@id`) into a DataFrame for analysis. 

Record sets and fields are referenced exclusively by their `@id`, as required by the Croissant schema and this notebook's usage style.

In [ ]:
# Get all record set @id's
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]
print('Record set @id list:', record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    # Read records from each record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}.")

# For demonstration, pick the first record set for EDA
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print('Working with main record set @id:', main_record_set_id)
    print('Columns:', dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Below we demonstrate common data processing steps, referencing fields by their Croissant `@id`. 

We'll:
- Select a numeric field (e.g., normalized abundance, molecular weight, or peptide count as referenced by their IDs)
- Filter by a threshold
- Normalize
- Optionally, group or summarize data by a categorical field (e.g., protein description, or accession/protein ID)

In [ ]:
# Choose numeric field(s) and group field by their @id
# These IDs should be referenced from the schema: for demonstration, we assume the following typical protein data fields
numeric_field_id = None
group_field_id = None

sample_df = dataframes[main_record_set_id]

# Attempt to auto-detect a numeric field
for c in sample_df.columns:
    if sample_df[c].dtype in [int, float] or pd.api.types.is_numeric_dtype(sample_df[c]):
        numeric_field_id = c
        break

# Attempt to auto-detect a group (categorical) field
for c in sample_df.columns:
    if sample_df[c].dtype == object and c != numeric_field_id:
        group_field_id = c
        break

print(f'Numeric field candidate: {numeric_field_id}')
print(f'Group field candidate: {group_field_id}')

# Set a threshold for the numeric field (arbitrarily 10)
threshold = 10

if numeric_field_id is not None:
    filtered_df = sample_df[sample_df[numeric_field_id].astype(float) > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the field
    _norm_col = f'{numeric_field_id}_normalized'
    filtered_df[_norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, _norm_col]].head())

    # If group field found, group and show mean
    if group_field_id and group_field_id in filtered_df:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id} and mean (for numeric columns):")
        display(grouped_df.head())

## 5. Visualization
Visualize numeric fields, distribution, or relationships, referencing the appropriate `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id].astype(float), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in filtered_df:
        # Show boxplot by group
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.xticks(rotation=45)
        plt.title(f'{numeric_field_id} distribution by {group_field_id}')
        plt.show()

## 6. Conclusion

- We loaded and explored the FAIR² mass spectrometry dataset using `mlcroissant`, accessing data elements by `@id` as prescribed by the Croissant schema.
- In this notebook, we listed available record sets and fields, extracted the main table into a DataFrame, filtered and normalized a numeric field, and visualized their distributions.

### Next Steps
- Use field `@id`s from the schema for more targeted analysis or integration.
- Combine with protein annotation or sample metadata for advanced ML tasks.